In [0]:
from pyspark.sql import functions as F
from config import TABLES

In [0]:
def log_pipeline_metric(pipeline_name, df):
    metrics_df = spark.createDataFrame(
        [(pipeline_name, df.count())],
        ["pipeline", "row_count"]
    ).withColumn("processed_at", F.current_timestamp())

    metrics_df.write.mode("append").saveAsTable("dbw_retails.monitoring.pipeline_metrics")

In [0]:
df_bronze = spark.read.table(TABLES["bronze_eventhub"])
df_silver = spark.read.table(TABLES["silver"])

In [0]:
print("Bronze count:", df_bronze.count())
print("Silver count:", df_silver.count())

Bronze count: 35000
Silver count: 28998


In [0]:
nulls = df_silver.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_silver.columns
])

In [0]:
display(nulls)

invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,invoice_ts,invoice_dt,_ingest_ts,_source_file,revenue,year,month,day
0,0,0,0,0,0,131763,0,0,0,0,19311,0,0,0,519609


In [0]:
dupes = (
    df_silver.groupBy("invoice_no","stock_code","customer_id","invoice_date")
    .count()
    .filter("count > 1")
)

In [0]:
print("Duplicate rows:", dupes.count())

Duplicate rows: 0


In [0]:
revenue_total = df_silver.agg(F.sum("revenue")).collect()[0][0]

In [0]:
print("Total revenue:", revenue_total)

Total revenue: 731260.3400000035
